In [ ]:
# Cell 1: Mount Google Drive & setup working directories
import os
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DIR = Path('/content/drive/MyDrive/quantroapex')
else:
    # Local fallback for smoke testing
    DRIVE_DIR = Path('./drive_quantroapex').resolve()

DATA_DIR = DRIVE_DIR / 'data'
CHECKPOINT_DIR = DRIVE_DIR / 'checkpoints'
REPORTS_DIR = DRIVE_DIR / 'reports'

for d in [DATA_DIR, CHECKPOINT_DIR, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f'Mounted storage directories initialized at: {DRIVE_DIR}')


In [ ]:
# Cell 2: Clone repository at pinned commit & verify GPU runtime
import sys
from pathlib import Path

PINNED_COMMIT = '70d70367eed3d5db719814b25474becb1bb27ccc'
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    repo_path = Path('/content/QuantroApex')
    if not repo_path.exists():
        !git clone -b trust-layer-and-ml https://github.com/Quantro-Software-Labs/QuantroApex.git /content/QuantroApex
        !cd /content/QuantroApex && git checkout {PINNED_COMMIT}
    %cd /content/QuantroApex/services/data-pipeline
    !pip install --no-deps -r requirements-train.txt || pip install -r requirements-train.txt
    if '/content/QuantroApex/services/data-pipeline' not in sys.path:
        sys.path.insert(0, '/content/QuantroApex/services/data-pipeline')
else:
    def find_pipeline_root():
        p = Path.cwd().resolve()
        for parent in [p] + list(p.parents):
            cand = parent / 'services' / 'data-pipeline'
            if cand.exists():
                return cand
            if (parent / 'ingestion').exists() and (parent / 'training').exists():
                return parent
        return p

    pipeline_root = find_pipeline_root()
    if str(pipeline_root) not in sys.path:
        sys.path.insert(0, str(pipeline_root))

# Guard check: ensure required validation files exist
repo_root = Path('/content/QuantroApex') if IN_COLAB else (pipeline_root.parent.parent if pipeline_root.name == 'data-pipeline' else pipeline_root)
required_files = [
    repo_root / 'docs' / 'ml_success_bar.md',
    repo_root / 'services' / 'data-pipeline' / 'evaluation' / 'metrics.py',
]
for rf in required_files:
    if not rf.exists():
        raise FileNotFoundError(f'CRITICAL: Required file {rf} is missing! Refusing to run old or unpinned code.')
print('Verified required validation files exist (docs/ml_success_bar.md and evaluation/metrics.py).')

import torch
print(f'PyTorch Version: {torch.__version__}')
print(f'CUDA Available: {torch.cuda.is_available()}')

if torch.cuda.is_available():
    print(f'GPU Name: {torch.cuda.get_device_name(0)}')
    print(f'CUDA Version: {torch.version.cuda}')
elif IN_COLAB:
    raise RuntimeError('No GPU available! Please enable GPU runtime: Runtime -> Change runtime type -> T4 GPU')
else:
    print('Running in local CPU smoke test environment.')


In [ ]:
# Cell 3: Download 1h and 4h spot klines to Drive (resume-safe, skip existing)
import os
from ingestion.binance_downloader import BinanceVisionDownloader

SMOKE_MODE = os.environ.get('SMOKE_MODE', '1') == '1'

downloader = BinanceVisionDownloader(data_dir=DATA_DIR)
if SMOKE_MODE:
    print('SMOKE_MODE=True: Downloading BTCUSDT 1h sample for quick verification...')
    downloader.download_all(symbols=['BTCUSDT'], intervals=['1h'], smoke=True)
else:
    print('Full Mode: Downloading 1h and 4h spot klines for BTCUSDT, ETHUSDT, SOLUSDT, BNBUSDT, XRPUSDT...')
    downloader.download_all(smoke=False)


In [ ]:
# Cell 4: Execute Purged Rolling Walk-Forward (Fold checkpoints saved to Drive)
import asyncio
from evaluation.walk_forward import run_walk_forward

print(f'Starting Purged Walk-Forward (Smoke={SMOKE_MODE}). Checkpoints saved to {CHECKPOINT_DIR}')
print('Walk-forward will log per-fold progress (fold k of N, elapsed time, ETA) as each fold executes.')

# Handle both Jupyter active event loop and standard terminal execution
try:
    import nest_asyncio
    nest_asyncio.apply()
except Exception:
    pass

try:
    loop = asyncio.get_running_loop()
    report = loop.run_until_complete(
        run_walk_forward(
            train_epochs=3 if SMOKE_MODE else 5,
            smoke=SMOKE_MODE,
            parquet_dir=DATA_DIR,
            checkpoint_dir=str(CHECKPOINT_DIR),
            allow_holdout=False,
        )
    )
except RuntimeError:
    report = asyncio.run(
        run_walk_forward(
            train_epochs=3 if SMOKE_MODE else 5,
            smoke=SMOKE_MODE,
            parquet_dir=DATA_DIR,
            checkpoint_dir=str(CHECKPOINT_DIR),
            allow_holdout=False,
        )
    )


In [ ]:
# Cell 5: Print Summary Table, Machine Verdict & Diagnostics
import json
import shutil

print('=' * 80)
print(f"MACHINE VERDICT: {report['verdict']}")
print(f"Positive Folds: {report['positive_folds']} ({report['positive_folds_pct']}%)")
print('=' * 80)
print()
print(report['markdown_table'])
print()

print("--- PRE-REGISTERED SUCCESS BAR CRITERIA ---")
for k, v in report['pre_registered_success_bar'].items():
    print(f'  {k}: {v}')

print("--- DIAGNOSTICS: TURNOVER & EXECUTION COSTS ---")
for k, v in report['diagnostics']['turnover_and_costs'].items():
    print(f'  {k}: {v}')

print("--- DIAGNOSTICS: MODEL CALIBRATION ---")
for b in report['diagnostics']['calibration']:
    print(f"  Bucket {b['bucket']}: Count={b['count']} | Pred Prob={b['mean_pred_prob']} | Realised Win Rate={b['realised_win_rate']:.1%}")

# Copy final reports to Drive reports dir
drive_json = REPORTS_DIR / 'walk_forward_report.json'
drive_md = REPORTS_DIR / 'report.md'
shutil.copy(CHECKPOINT_DIR / 'walk_forward_report.json', drive_json)
shutil.copy(CHECKPOINT_DIR / 'report.md', drive_md)
print(f'Final reports persisted to Drive: {drive_json} and {drive_md}')
